# 06. 결과 통합 및 Figure 생성

Gurobi / SA / QA 결과를 하나의 표로 합치고 Figure 1~7을 생성한다.

## 비교 원칙

- 모든 solver 결과는 **원래 CFLP objective space**로 decode하여 비교한다. QUBO energy만 비교하지 않는다.
- true optimality gap의 기준은 각 formulation의 Gurobi 최적값이다.

$$Gap_{true} = \frac{Obj_{solver} - Obj_{Gurobi}}{|Obj_{Gurobi}|} \times 100$$

- SA/QA의 목적값으로는 **feasible sample 중 최선**을 사용한다. infeasible한 해의 목적값은 제약을 지키지 않아 얻은 것이므로 solution quality로 볼 수 없다.
- QUBO energy gap도 별도로 기록하되, 핵심 성능 비교에는 true-optimum gap을 우선한다.

In [1]:
# 프로젝트 루트를 import 경로에 추가한다.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import load_config, resolve_path

config = load_config(PROJECT_ROOT / "config" / "experiment_config.yaml")
DATA_DIR = resolve_path(config, "data_dir")
RAW_DIR = resolve_path(config, "raw_dir")
PROCESSED_DIR = resolve_path(config, "processed_dir")
FIGURE_DIR = resolve_path(config, "figure_dir")
SOLUTION_DIR = RAW_DIR / "solutions"
SOLUTION_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
print("설정 로드 완료:", len(config["instances"]), "개 instance")


설정 로드 완료: 4 개 instance


In [2]:
from src.persistence import load_table, save_table

gurobi_results = load_table(RAW_DIR, "gurobi_results.csv")
qubo_stats = load_table(RAW_DIR, "qubo_stats.csv")
sa_results = load_table(RAW_DIR, "sa_results.csv")
try:
    qa_results = load_table(RAW_DIR, "qa_results.csv")
except FileNotFoundError:
    qa_results = pd.DataFrame()
    print("QA 결과 파일이 없습니다. QA 없이 통합합니다.")
print(
    "불러온 행 수:",
    len(gurobi_results), len(qubo_stats), len(sa_results), len(qa_results),
)

불러온 행 수: 12 8 16 16


## 참조 최적값 정리

- SS의 기준: Gurobi `SS`
- MS의 기준: Gurobi `MS` (원래 연속 formulation)
- 추가로 `MS-int-q`를 기록하여 1 unit 이산화 손실을 분리한다.

In [3]:
reference = gurobi_results.pivot_table(
    index=["instance", "size"], columns="gurobi_model", values="objective"
).reset_index()
reference = reference.rename(
    columns={"SS": "ref_SS", "MS": "ref_MS", "MS-int-q": "ref_MS_INT"}
)
reference["discretization_loss_percent"] = (
    (reference["ref_MS_INT"] - reference["ref_MS"])
    / reference["ref_MS"].abs()
    * 100
).round(6)
reference

gurobi_model,instance,size,ref_MS,ref_MS_INT,ref_SS,discretization_loss_percent
0,15x15,15,7909.2986,7909.2986,8489.1855,0.0
1,4x4,4,2416.8364,2416.8364,2416.8364,0.0
2,6x6,6,3968.8513,3968.8513,4886.7940,0.0
3,8x8,8,3626.6311,3626.6311,3952.5153,0.0


In [4]:
def reference_objective(row: pd.Series) -> float:
    """formulation에 맞는 Gurobi true optimum을 반환한다."""
    matched = reference[reference["instance"] == row["instance"]].iloc[0]
    return float(matched["ref_SS" if row["formulation"] == "SS" else "ref_MS"])


def build_solver_rows(frame: pd.DataFrame, solver: str) -> pd.DataFrame:
    """SA/QA 결과를 공통 스키마로 변환한다."""
    if frame.empty:
        return pd.DataFrame()
    rows = frame.copy()
    rows["solver"] = solver
    rows["gurobi_optimum"] = rows.apply(reference_objective, axis=1)
    # feasible 해가 없으면 목적값을 NaN으로 둔다 (성능으로 인정하지 않는다).
    rows["objective"] = rows["best_feasible_objective"]
    rows["is_feasible"] = rows["best_feasible_objective"].notna()
    rows["total_violation"] = rows["best_total_violation"]
    return rows


solver_rows = pd.concat(
    [build_solver_rows(sa_results, "SA"), build_solver_rows(qa_results, "QA")],
    ignore_index=True,
)
solver_rows["true_gap_percent"] = (
    (solver_rows["objective"] - solver_rows["gurobi_optimum"])
    / solver_rows["gurobi_optimum"].abs()
    * 100
)
print(len(solver_rows), "개 solver 결과")

32 개 solver 결과


In [5]:
gurobi_rows = []
for formulation in ("SS", "MS"):
    model_name = "SS" if formulation == "SS" else "MS"
    subset = gurobi_results[gurobi_results["gurobi_model"] == model_name].copy()
    subset["formulation"] = formulation
    subset["solver"] = "Gurobi"
    subset["gurobi_optimum"] = subset["objective"]
    subset["true_gap_percent"] = 0.0
    subset["is_feasible"] = True
    subset["total_violation"] = 0.0
    subset["feasible_fraction"] = 1.0
    gurobi_rows.append(subset)
gurobi_rows = pd.concat(gurobi_rows, ignore_index=True)

all_results = pd.concat([gurobi_rows, solver_rows], ignore_index=True)
all_results = all_results.merge(
    qubo_stats.drop(columns=["size"]), on=["instance", "formulation"], how="left"
)
if "embedding_status" not in all_results.columns:
    all_results["embedding_status"] = np.nan
all_results["embedding_status"] = all_results["embedding_status"].fillna(
    all_results["solver"].map({"Gurobi": "N/A", "SA": "N/A"})
)
save_table(all_results, PROCESSED_DIR, "all_results.csv")
print("저장:", PROCESSED_DIR / "all_results.csv")
all_results[["instance", "formulation", "solver", "status", "objective", "true_gap_percent"]]

저장: C:\Users\User\Desktop\KMJ\Study\Quantum\cflp_formulation\results\processed\all_results.csv


,instance,formulation,solver,status,objective,true_gap_percent
0,4x4,SS,Gurobi,OPTIMAL,2416.8364,0.000000
1,6x6,SS,Gurobi,OPTIMAL,4886.7940,0.000000
2,8x8,SS,Gurobi,OPTIMAL,3952.5153,0.000000
3,15x15,SS,Gurobi,OPTIMAL,8489.1855,0.000000
4,4x4,MS,Gurobi,OPTIMAL,2416.8364,0.000000
5,6x6,MS,Gurobi,OPTIMAL,3968.8513,0.000000
6,8x8,MS,Gurobi,OPTIMAL,3626.6311,0.000000
7,15x15,MS,Gurobi,OPTIMAL,7909.2986,0.000000
8,4x4,SS,SA,OK,2593.0879,7.292653
9,4x4,SS,SA,OK,2416.8364,0.000000


## 최종 비교표

요청된 형태의 `formulation x instance x solver` 비교표이다.

In [6]:
comparison = all_results.pivot_table(
    index=["formulation", "size", "instance"],
    columns="solver",
    values=["objective", "true_gap_percent", "runtime"],
    dropna=False,
).sort_index(level=["formulation", "size"])
save_table(comparison.reset_index(), PROCESSED_DIR, "comparison_table.csv")
comparison.round(3)

objective                      runtime                true_gap_percent                
solver                       Gurobi        QA         SA  Gurobi     QA      SA           Gurobi      QA      SA
formulation size instance                                                                                       
MS          4    15x15          NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
                 4x4       2416.836       NaN   3350.999   0.009  1.602   2.854              0.0     NaN  38.652
                 6x6            NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
                 8x8            NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
            6    15x15          NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
                 4x4            NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
                 6x6       3968.851       NaN   5562.173   0.016    NaN   8.001              0.0     NaN  40.146
                 8x8            NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
            8    15x15          NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
                 4x4            NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
                 6x6            NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
                 8x8       3626.631       NaN   6547.024   0.002    NaN  14.661              0.0     NaN  80.526
            15   15x15     7909.299       NaN  14491.096   0.021    NaN  76.201              0.0     NaN  83.216
                 4x4            NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
                 6x6            NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
                 8x8            NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
SS          4    15x15          NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
                 4x4       2416.836  3078.054   2504.962   0.006  1.227   0.494              0.0  27.359   3.646
                 6x6            NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
                 8x8            NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
            6    15x15          NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
                 4x4            NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
                 6x6       4886.794       NaN   5850.012   0.001  1.205   0.890              0.0     NaN  19.711
                 8x8            NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
            8    15x15          NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
                 4x4            NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
                 6x6            NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
                 8x8       3952.515       NaN   5617.319   0.006  1.350   1.450              0.0     NaN  42.120
            15   15x15     8489.186       NaN        NaN   0.100  2.011   4.538              0.0     NaN     NaN
                 4x4            NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
                 6x6            NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN
                 8x8            NaN       NaN        NaN     NaN    NaN     NaN              NaN     NaN     NaN

## QA embedding 표

Figure 7의 기초 자료이다.

In [7]:
if qa_results.empty:
    embedding_table = qubo_stats[["instance", "size", "formulation"]].copy()
    embedding_table["embedding_status"] = "NOT_ATTEMPTED"
else:
    embedding_table = qa_results[["instance", "size", "formulation", "status"]].copy()
    embedding_table = embedding_table.rename(columns={"status": "embedding_status"})
save_table(embedding_table, PROCESSED_DIR, "embedding_table.csv")
embedding_table

,instance,size,formulation,embedding_status
0,4x4,4,SS,OK
1,4x4,4,SS,OK
2,4x4,4,MS,OK
3,4x4,4,MS,OK
4,6x6,6,SS,OK
5,6x6,6,SS,OK
6,6x6,6,MS,NOT_EMBEDDABLE
7,6x6,6,MS,NOT_EMBEDDABLE
8,8x8,8,SS,OK
9,8x8,8,SS,OK


## Figure 1 ~ 7 생성

In [8]:
from src.plotting import generate_all_figures

paths = generate_all_figures(all_results, qubo_stats, embedding_table, FIGURE_DIR)
for path in paths:
    print("저장:", path.name)

저장: figure1_objective_comparison.png
저장: figure2_true_gap.png
저장: figure3_runtime.png
저장: figure4_qubo_size.png
저장: figure5_feasibility.png
저장: figure6_coefficient_range.png
저장: figure7_embedding.png


In [ ]:
from IPython.display import Image, display

for path in paths:
    display(Image(filename=str(path)))

## 결과 요약

아래 요약은 관측된 수치만 기술한다. runtime이 짧다는 이유만으로 우열을 단정하지 않으며, solution quality / exactness / feasibility / classical runtime / QPU runtime / scalability / QUBO size / embedding feasibility를 구분해서 본다. 또한 이 실험은 QA parameter tuning 실험이 아니므로 특정 QA 파라미터가 최적이라는 결론을 내리지 않는다.

In [ ]:
lines = []
lines.append("[QUBO 크기]")
for _, row in qubo_stats.sort_values(["size", "formulation"]).iterrows():
    lines.append(
        f"  {row['instance']:>6s} {row['formulation']}: "
        f"변수 {int(row['qubo_variables']):5d}, 항 {int(row['qubo_terms']):7d}, "
        f"계수 범위 {row['qubo_range']:.3e}"
    )
lines.append("")
lines.append("[이산화 손실 (MS-int-q vs MS)]")
for _, row in reference.iterrows():
    lines.append(
        f"  {row['instance']:>6s}: {row['discretization_loss_percent']:.6f}%"
    )
lines.append("")
lines.append("[SA / QA true optimality gap]")
for _, row in solver_rows.sort_values(["solver", "formulation", "size"]).iterrows():
    gap = row["true_gap_percent"]
    status = str(row["status"])
    if status != "OK":
        # 실행 자체를 못 한 경우와 '실행했지만 feasible 해가 없음'을
        # 반드시 구분해서 적는다.
        detail = f"미실행 ({status})"
    elif pd.isna(gap):
        detail = "실행했으나 feasible 해 없음"
    else:
        detail = (
            f"gap {gap:8.3f}%, feasible 비율 "
            f"{row.get('feasible_fraction', float('nan')):.3f}"
        )
    lines.append(
        f"  {row['solver']:>2s} {row['formulation']} {row['instance']:>6s}: {detail}"
    )
summary_text = "\n".join(lines)
(PROCESSED_DIR / "summary.txt").write_text(summary_text, encoding="utf-8")
print(summary_text)